In [1]:
SYMBOL = "BTCUSDT"
TARGET_HORIZON = 6
INTERVAL = "5m"
MODEL_TYPE = "rf"

In [2]:
# Parameters
SYMBOL = "MATICUSDT"
INTERVAL = "5m"
TARGET_HORIZON = 6
MODEL_TYPE = "rf"


In [3]:
import os
import time
import json
import joblib
import pandas as pd
import numpy as np
import optuna
from optuna.pruners import MedianPruner
from functools import partial
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)
from features import add_features
from constants import DATA_DIR, MODEL_DIR
from utils import time_split
from models import tune_selected_features_only , make_bucket_table, fit_final_model

/home/rachmiel/quant/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
MODEL_DIR = os.path.join(MODEL_DIR, MODEL_TYPE)
PARQUET_PATH = f"{DATA_DIR}/{SYMBOL}_{INTERVAL}.parquet"

os.makedirs(MODEL_DIR, exist_ok=True)

In [5]:
model_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_model.joblib")

features_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_cols.json")
meta_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_meta.json")
fi_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_importance.csv")
pred_path = os.path.join(MODEL_DIR, f"{SYMBOL}__{TARGET_HORIZON}_predictions.csv")

In [6]:
df = pd.read_parquet(PARQUET_PATH)
print(f"[info] raw rows: {len(df):,}")

# add features + target
df, feature_cols = add_features(df, TARGET_HORIZON)

[info] raw rows: 1,000


In [7]:
df.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,num_trades,taker_buy_base_asset_volume,...,dow_cos,dom_sin,dom_cos,month_sin,month_cos,macd,macd_signal,macd_hist,atr_14,atr_norm
0,2024-09-06 15:40:00+00:00,0.3662,0.3679,0.3660,0.3677,172085.1,2024-09-06 15:44:59.999000+00:00,63124.74949,184,109146.8,...,-0.900969,0.937752,0.347305,-1.0,-1.836970e-16,0.000000,0.000000,0.000000,NaN,NaN
1,2024-09-06 15:45:00+00:00,0.3679,0.3683,0.3666,0.3667,236756.7,2024-09-06 15:49:59.999000+00:00,87019.72975,342,81575.3,...,-0.900969,0.937752,0.347305,-1.0,-1.836970e-16,-0.000022,-0.000012,-0.000010,NaN,NaN
2,2024-09-06 15:50:00+00:00,0.3669,0.3670,0.3655,0.3668,141178.7,2024-09-06 15:54:59.999000+00:00,51695.07204,216,90499.5,...,-0.900969,0.937752,0.347305,-1.0,-1.836970e-16,-0.000025,-0.000018,-0.000008,NaN,NaN
3,2024-09-06 15:55:00+00:00,0.3671,0.3671,0.3655,0.3663,93002.4,2024-09-06 15:59:59.999000+00:00,34067.68869,162,48157.3,...,-0.900969,0.937752,0.347305,-1.0,-1.836970e-16,-0.000044,-0.000027,-0.000018,NaN,NaN
4,2024-09-06 16:00:00+00:00,0.3661,0.3665,0.3651,0.3654,99800.5,2024-09-06 16:04:59.999000+00:00,36492.59671,207,55007.2,...,-0.900969,0.937752,0.347305,-1.0,-1.836970e-16,-0.000089,-0.000045,-0.000044,NaN,NaN


In [8]:
target_col = f"target_{TARGET_HORIZON}"
ret_col = f"target_ret_fwd_{TARGET_HORIZON}"

model_df = df[["open_time"] + feature_cols + [target_col, ret_col]].copy()

# Remove:
# early rows where rolling features don’t exist yet
# rows where z-scores / ratios blew up
# rows where target is NaN (due to future shift)
model_df = model_df.replace([np.inf, -np.inf], np.nan)
model_df = model_df.dropna(subset=feature_cols + [target_col, ret_col])

print(f"[info] usable rows after features: {len(model_df):,}")

train_df, test_df = time_split(model_df, train_frac=0.8)

# Further split the training set into train/valid for Optuna
optuna_train_df, valid_df = time_split(train_df, train_frac=0.8)

X_train = optuna_train_df[feature_cols]
y_train = optuna_train_df[target_col]

X_valid = valid_df[feature_cols]
y_valid = valid_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]
fwd_ret = test_df[ret_col]

train_start_time = pd.to_datetime(train_df["open_time"].iloc[0], utc=True)
train_end_time = pd.to_datetime(train_df["open_time"].iloc[-1], utc=True)

val_start_time = pd.to_datetime(valid_df["open_time"].iloc[0], utc=True)
val_end_time = pd.to_datetime(valid_df["open_time"].iloc[-1], utc=True)

test_start_time = pd.to_datetime(test_df["open_time"].iloc[0], utc=True)
test_end_time = pd.to_datetime(test_df["open_time"].iloc[-1], utc=True)

print(f"[info] optuna train rows: {len(optuna_train_df):,}")
print(f"[info] valid rows:        {len(valid_df):,}")
print(f"[info] test rows:         {len(test_df):,}")

[info] usable rows after features: 921
[info] optuna train rows: 588
[info] valid rows:        148
[info] test rows:         185


In [9]:
results = tune_selected_features_only(
    model_type=MODEL_TYPE,
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
    n_trials=100,
    objective_metric="roc_auc",
    top_k=25,
)

print(results["selected_features"])
print(results["feature_importance"].head(30))

[I 2026-03-22 18:28:51,172] A new study created in memory with name: no-name-57ab4359-0231-4c71-9c28-3f1d133b6b8a


[I 2026-03-22 18:28:51,868] Trial 0 finished with value: 0.48456621004566214 and parameters: {'n_estimators': 400, 'max_depth': 12, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': 0.5, 'bootstrap': True, 'class_weight': None, 'criterion': 'log_loss'}. Best is trial 0 with value: 0.48456621004566214.


[I 2026-03-22 18:28:52,501] Trial 1 finished with value: 0.5488584474885846 and parameters: {'n_estimators': 500, 'max_depth': 7, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_features': 0.8, 'bootstrap': False, 'class_weight': 'balanced_subsample', 'criterion': 'log_loss'}. Best is trial 1 with value: 0.5488584474885846.


[I 2026-03-22 18:28:53,914] Trial 2 finished with value: 0.4955251141552512 and parameters: {'n_estimators': 800, 'max_depth': 11, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'bootstrap': True, 'class_weight': None, 'criterion': 'log_loss'}. Best is trial 1 with value: 0.5488584474885846.


[I 2026-03-22 18:28:55,091] Trial 3 finished with value: 0.49077625570776257 and parameters: {'n_estimators': 700, 'max_depth': 12, 'min_samples_split': 11, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'bootstrap': True, 'class_weight': None, 'criterion': 'log_loss'}. Best is trial 1 with value: 0.5488584474885846.


...

[I 2026-03-22 18:29:47,068] Trial 96 pruned. 


[I 2026-03-22 18:29:47,472] Trial 97 pruned. 


[I 2026-03-22 18:29:47,892] Trial 98 finished with value: 0.5552511415525114 and parameters: {'n_estimators': 300, 'max_depth': 8, 'min_samples_split': 7, 'min_samples_leaf': 2, 'max_features': 0.8, 'bootstrap': False, 'class_weight': 'balanced', 'criterion': 'log_loss'}. Best is trial 80 with value: 0.5567123287671233.


[I 2026-03-22 18:29:48,343] Trial 99 pruned. 


['atr_norm', 'range_15', 'imbalance_15', 'trend_x_imb', 'range_5', 'mom_60', 'dist_ma_30', 'vol_30', 'trend_strength', 'hour_cos', 'vol_15', 'vol_regime_ratio', 'vol_5', 'vol_ratio_5_30', 'dist_ma_15_z', 'mom_30', 'imbalance_5', 'dist_ma_15', 'num_trades_mom_5', 'bar_range', 'mr_x_vol', 'mom_15', 'macd_hist', 'mom_10', 'dist_ma_5']
feature
atr_norm            0.047019
range_15            0.043905
imbalance_15        0.042456
trend_x_imb         0.040508
range_5             0.040420
mom_60              0.037615
dist_ma_30          0.035461
vol_30              0.034444
trend_strength      0.032156
hour_cos            0.031749
vol_15              0.030689
vol_regime_ratio    0.030579
vol_5               0.030499
vol_ratio_5_30      0.029784
dist_ma_15_z        0.029434
mom_30              0.027781
imbalance_5         0.027778
dist_ma_15          0.026308
num_trades_mom_5    0.025944
bar_range           0.023847
mr_x_vol            0.023750
mom_15              0.023647
macd_hist           

In [10]:
artifacts = fit_final_model(
    model_type=MODEL_TYPE,
    best_params=results["best_params"],
    selected_features=results["selected_features"],
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
)

calibrator = artifacts["calibrator"]
base_model = artifacts["base_model"]
selected_features = artifacts["selected_features"]

In [11]:
X_train_sel = X_train[selected_features].copy()
X_valid_sel = X_valid[selected_features].copy()
X_train_full_sel = pd.concat([X_train_sel, X_valid_sel], axis=0)

X_test_sel = X_test[selected_features].copy()
y_train_full = pd.concat([y_train, y_valid], axis=0)

calibrator = artifacts["calibrator"]

train_pred = calibrator.predict_proba(X_train_full_sel)[:, 1]
test_pred = calibrator.predict_proba(X_test_sel)[:, 1]

In [12]:
train_pred_label = (train_pred >= 0.5).astype(int)
test_pred_label = (test_pred >= 0.5).astype(int)

print("[eval] computing metrics...")

train_auc = roc_auc_score(y_train_full, train_pred)
test_auc = roc_auc_score(y_test, test_pred)

train_pr_auc = average_precision_score(y_train_full, train_pred)
test_pr_auc = average_precision_score(y_test, test_pred)

train_logloss = log_loss(y_train_full, np.clip(train_pred, 1e-8, 1 - 1e-8))
test_logloss = log_loss(y_test, np.clip(test_pred, 1e-8, 1 - 1e-8))

train_brier = brier_score_loss(y_train_full, train_pred)
test_brier = brier_score_loss(y_test, test_pred)

train_acc = accuracy_score(y_train_full, train_pred_label)
test_acc = accuracy_score(y_test, test_pred_label)

train_precision = precision_score(y_train_full, train_pred_label, zero_division=0)
test_precision = precision_score(y_test, test_pred_label, zero_division=0)

train_recall = recall_score(y_train_full, train_pred_label, zero_division=0)
test_recall = recall_score(y_test, test_pred_label, zero_division=0)

train_f1 = f1_score(y_train_full, train_pred_label, zero_division=0)
test_f1 = f1_score(y_test, test_pred_label, zero_division=0)

print("\n===== RESULTS =====")
print(f"Train ROC AUC:   {train_auc:.6f}")
print(f"Test ROC AUC:    {test_auc:.6f}")
print(f"Train PR AUC:    {train_pr_auc:.6f}")
print(f"Test PR AUC:     {test_pr_auc:.6f}")
print(f"Train Log Loss:  {train_logloss:.6f}")
print(f"Test Log Loss:   {test_logloss:.6f}")
print(f"Train Brier:     {train_brier:.6f}")
print(f"Test Brier:      {test_brier:.6f}")
print(f"Train Accuracy:  {train_acc:.6f}")
print(f"Test Accuracy:   {test_acc:.6f}")
print(f"Train Precision: {train_precision:.6f}")
print(f"Test Precision:  {test_precision:.6f}")
print(f"Train Recall:    {train_recall:.6f}")
print(f"Test Recall:     {test_recall:.6f}")
print(f"Train F1:        {train_f1:.6f}")
print(f"Test F1:         {test_f1:.6f}")

[eval] computing metrics...

===== RESULTS =====
Train ROC AUC:   0.949779
Test ROC AUC:    0.518631
Train PR AUC:    0.956513
Test PR AUC:     0.469352
Train Log Loss:  0.550428
Test Log Loss:   0.704030
Train Brier:     0.180111
Test Brier:      0.255378
Train Accuracy:  0.869565
Test Accuracy:   0.454054
Train Precision: 0.836406
Test Precision:  0.413223
Train Recall:    0.935567
Test Recall:     0.625000
Train F1:        0.883212
Test F1:         0.497512


In [13]:
eval_df = pd.DataFrame({
    "pred": test_pred,
    "y_cls": y_test.values,
    "fwd_ret": fwd_ret.values,   # continuous realised return
})

eval_df["pred_bin"] = pd.qcut(eval_df["pred"], 10, duplicates="drop")
bucket_stats = eval_df.groupby("pred_bin")["fwd_ret"].agg(["mean", "count", "std"])
print(bucket_stats)

                    mean  count       std
pred_bin                                 
(0.372, 0.421]  0.002290     19  0.005924
(0.421, 0.465]  0.000363     18  0.005296
(0.465, 0.489]  0.003061     19  0.006545
(0.489, 0.514] -0.000547     18  0.002977
(0.514, 0.532] -0.000138     19  0.002710
(0.532, 0.543] -0.000217     18  0.002568
(0.543, 0.555] -0.000358     18  0.002278
(0.555, 0.57]  -0.000810     19  0.003274
(0.57, 0.61]   -0.000785     18  0.003443
(0.61, 0.651]  -0.000486     19  0.003601


/tmp/ipykernel_1013531/1883822384.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket_stats = eval_df.groupby("pred_bin")["fwd_ret"].agg(["mean", "count", "std"])


In [14]:
top_decile_threshold = float(np.quantile(test_pred, 0.9))
bottom_decile_threshold = float(np.quantile(test_pred, 0.1))

top_decile_mean_ret = float(eval_df.loc[eval_df["pred"] >= top_decile_threshold, "fwd_ret"].mean())
bottom_decile_mean_ret = float(eval_df.loc[eval_df["pred"] <= bottom_decile_threshold, "fwd_ret"].mean())
overall_mean_ret = float(eval_df["fwd_ret"].mean())

signal_threshold = 0.6
signal_rate = float((eval_df["pred"] >= signal_threshold).mean())
signal_mean_ret = float(eval_df.loc[eval_df["pred"] >= signal_threshold, "fwd_ret"].mean())

In [15]:
# save predictions
out = test_df[["open_time", target_col]].copy()
out["prediction"] = test_pred
out.to_csv(pred_path, index=False)
print(f"\n[saved] predictions -> {pred_path}")


[saved] predictions -> models/rf/MATICUSDT__6_predictions.csv


In [16]:
# save model
joblib.dump(artifacts, model_path)

# save feature columns
with open(features_path, "w") as f:
    json.dump(selected_features, f, indent=2)

# save feature importance
results["feature_importance"].to_csv(fi_path, header=["importance"])

# save metadata
meta = {
    "symbol": SYMBOL,
    "target_horizon": int(TARGET_HORIZON),
    "target_col": target_col,
    "model_type": MODEL_TYPE,
    "study_best_value": float(results["study"].best_value),
    "model_params": results["best_params"],
    "n_features": int(len(selected_features)),
    "feature_cols_path": str(features_path),
    "model_path": str(model_path),
    "feature_importance_path": str(fi_path) if fi_path is not None else None,
    "train_auc": float(train_auc),
    "test_auc": float(test_auc),
    "train_pr_auc": float(train_pr_auc),
    "test_pr_auc": float(test_pr_auc),
    "train_logloss": float(train_logloss),
    "test_logloss": float(test_logloss),
    "train_brier": float(train_brier),
    "test_brier": float(test_brier),
    "train_accuracy": float(train_acc),
    "test_accuracy": float(test_acc),
    "train_precision": float(train_precision),
    "test_precision": float(test_precision),
    "train_recall": float(train_recall),
    "test_recall": float(test_recall),
    "train_f1": float(train_f1),
    "test_f1": float(test_f1),
    "test_top_decile_threshold": top_decile_threshold,
    "test_bottom_decile_threshold": bottom_decile_threshold,
    "test_top_decile_mean_fwd_ret": top_decile_mean_ret,
    "test_bottom_decile_mean_fwd_ret": bottom_decile_mean_ret,
    "test_overall_mean_fwd_ret": overall_mean_ret,
    "test_signal_threshold": signal_threshold,
    "test_signal_rate": signal_rate,
    "test_signal_mean_fwd_ret": signal_mean_ret,
    "train_start_time": pd.Timestamp(train_start_time).isoformat(),
    "train_end_time": pd.Timestamp(train_end_time).isoformat(),
    "val_start_time": pd.Timestamp(val_start_time).isoformat(),
    "val_end_time": pd.Timestamp(val_end_time).isoformat(),
    "test_start_time": pd.Timestamp(test_start_time).isoformat(),
    "test_end_time": pd.Timestamp(test_end_time).isoformat(),
    "train_positive_rate": float(y_train_full.mean()),
    "test_positive_rate": float(y_test.mean()),
    "test_pred_mean": float(np.mean(test_pred)),
    "test_pred_std": float(np.std(test_pred)),
    "test_pred_p10": float(np.quantile(test_pred, 0.10)),
    "test_pred_p50": float(np.quantile(test_pred, 0.50)),
    "test_pred_p90": float(np.quantile(test_pred, 0.90)),
}

with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"[saved] model -> {model_path}")
print(f"[saved] features -> {features_path}")
print(f"[saved] feature importance -> {fi_path}")
print(f"[saved] metadata -> {meta_path}")

[saved] model -> models/rf/MATICUSDT__h6_model.joblib
[saved] features -> models/rf/MATICUSDT__h6_feature_cols.json
[saved] feature importance -> models/rf/MATICUSDT__h6_feature_importance.csv
[saved] metadata -> models/rf/MATICUSDT__h6_meta.json
